In [1]:
%cd ..

/Users/whisky-pizza/Desktop/projects/freshflow_experiment_inventory_optimisation


In [2]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import retail_configs


In [3]:
train = pd.read_parquet("data/train_small.parquet")
val   = pd.read_parquet("data/val_small.parquet")
eval_ = pd.read_parquet("data/eval_small.parquet")


In [17]:
TAU = retail_configs.TAU
CAT_COLS = retail_configs.CAT_COLS
FEATURES = retail_configs.FEATURES
TARGET = retail_configs.TARGET

In [5]:
cat_types = {c: pd.CategoricalDtype(sorted(train[c].unique())) for c in CAT_COLS}


In [6]:
cat_types

{'store_id': CategoricalDtype(categories=[  1,   2,   5,   6,   9,  11,  17,  18,  19,  21,
                   ...
                   877, 881, 882, 883, 885, 890, 894, 895, 896, 897],
 , ordered=False, categories_dtype=int64),
 'product_id': CategoricalDtype(categories=[  4,  18,  19,  20,  21,  23,  26,  41,  50,  64,
                   ...
                   740, 774, 775, 791, 792, 796, 797, 804, 810, 843],
 , ordered=False, categories_dtype=int64),
 'first_category_id': CategoricalDtype(categories=[4, 16, 29], ordered=False, categories_dtype=int64),
 'third_category_id': CategoricalDtype(categories=[  1,  10,  58,  60,  77,  81,  82,  94,  95,  96,  97,  98,
                    99, 100, 101, 103, 104, 105, 106, 131, 142, 149, 156, 157,
                   161, 163, 167, 168, 174, 179, 180, 187, 216, 231],
 , ordered=False, categories_dtype=int64)}

In [7]:
def prepare(df):
    df = df.copy()
    df["weekday"] = df["dt"].dt.weekday
    for c in retail_configs.CAT_COLS:
        df[c] = df[c].astype(cat_types[c])
    return df

train, val, eval_ = prepare(train), prepare(val), prepare(eval_)

/var/folders/d3/5wvh4m416kq1rdkwy9t366g00000gp/T/ipykernel_85435/4251020466.py:3: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["weekday"] = df["dt"].dt.weekday
/var/folders/d3/5wvh4m416kq1rdkwy9t366g00000gp/T/ipykernel_85435/4251020466.py:3: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["weekday"] = df["dt"].dt.weekday
/var/folders/d3/5wvh4m416kq1rdkwy9t366g00000gp/T/ipykernel_85435/4251020466.py:3: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["weekday"] = df["dt"].dt.weekday


1. lightgbm quantile model 

In [10]:
val_dates = np.sort(val["dt"].unique())
es_dates, cal_dates = val_dates[:7], val_dates[7:]

val_es  = val[val["dt"].isin(es_dates)]    # early stopping
val_cal = val[val["dt"].isin(cal_dates)]   # conformal calibration

In [ ]:
lgbm_models, lgbm_fit_time = {}, {}

for cat, tau in TAU.items():
    val_c = val_es[val_es["first_category_id"] == cat]
    model = lgb.LGBMRegressor(
        objective="quantile", alpha=tau,
        n_estimators=3000, learning_rate=0.05,
        num_leaves=63, min_child_samples=100,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, verbose=-1,
    )
    t0 = time.perf_counter()
    model.fit(train[FEATURES], train[TARGET],
              eval_X=[val_c[FEATURES]], eval_y=[val_c[TARGET]],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
    lgbm_fit_time[cat] = time.perf_counter() - t0
    lgbm_models[cat] = model

def predict_lgbm(df):
    q = pd.Series(np.nan, index=df.index)
    for cat, model in lgbm_models.items():
        rows = df["first_category_id"] == cat
        q[rows] = model.predict(df.loc[rows, FEATURES])
    return q.clip(lower=0)

eval_["q_lgbm"] = predict_lgbm(eval_)

/Users/whisky-pizza/Desktop/projects/freshflow_experiment_inventory_optimisation/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's quantile: 3.11129
[400]	valid_0's quantile: 3.11251
Early stopping, best iteration is:
[334]	valid_0's quantile: 3.1006


/Users/whisky-pizza/Desktop/projects/freshflow_experiment_inventory_optimisation/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[88]	valid_0's quantile: 1.11978
Training until validation scores don't improve for 100 rounds


/Users/whisky-pizza/Desktop/projects/freshflow_experiment_inventory_optimisation/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's quantile: 1.13045
Early stopping, best iteration is:
[101]	valid_0's quantile: 1.12555


2. conformal correction
For each category: on the calibration days, compute how far actual demand was above the predicted quantile, then take the τ-quantile of those errors as the correction.

In [12]:
def conformal_correction(y, q, tau):
    """Shift that makes P(y <= q + shift) ≈ tau on calibration data."""
    scores = y - q
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * tau) / n)   # finite-sample adjustment
    return float(np.quantile(scores, level))

val_cal = val_cal.copy()
val_cal["q_lgbm"] = predict_lgbm(val_cal)

corrections = {}
for cat, tau in TAU.items():
    d = val_cal[val_cal["first_category_id"] == cat]
    corrections[cat] = conformal_correction(d[TARGET].to_numpy(), d["q_lgbm"].to_numpy(), tau)
print(corrections)

eval_["q_lgbm_conf"] = eval_["q_lgbm"].copy()
for cat, shift in corrections.items():
    rows = eval_["first_category_id"] == cat
    eval_.loc[rows, "q_lgbm_conf"] = (eval_.loc[rows, "q_lgbm"] + shift).clip(lower=0)

{29: 2.357238341909403, 16: 1.1437281461621511, 4: 2.3833754914985827}


- compare 

In [13]:
def pinball(y, q, tau):
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))

rows = []
for cat, tau in TAU.items():
    d = eval_[eval_["first_category_id"] == cat]
    y = d[TARGET].to_numpy()
    for name, col in [("LightGBM", "q_lgbm"), ("LightGBM + conformal", "q_lgbm_conf")]:
        rows.append({
            "category": retail_configs.CATEGORY_CONFIG[cat]["category_name"],
            "tau": tau, "model": name,
            "coverage": (y <= d[col]).mean(),
            "pinball": pinball(y, d[col].to_numpy(), tau),
        })
pd.DataFrame(rows)

,category,tau,model,coverage,pinball
0,bread,0.60,LightGBM,0.481633,3.715040
1,bread,0.60,LightGBM + conformal,0.589116,3.565059
2,meat,0.75,LightGBM,0.706312,1.387084
3,meat,0.75,LightGBM + conformal,0.793355,1.393861
4,vegetables,0.88,LightGBM,0.787243,1.600206
5,vegetables,0.88,LightGBM + conformal,0.878113,1.495137


raw lightGBM is under predicting, could be an upward trend of sales entering summer, check that:

In [14]:
for name, df in [("train", train), ("val", val), ("eval", eval_)]:
    weekly = df.groupby([df["dt"].dt.isocalendar().week, "first_category_id"])[TARGET].mean()
    print(name, weekly.unstack().round(1).tail(3), sep="\n")

train
first_category_id     4   16    29
week                              
22                 13.3  8.7  19.5
23                 13.0  8.4  16.9
24                 15.6  8.9  24.3
val
first_category_id     4   16    29
week                              
24                 14.2  8.5  20.9
25                 16.2  9.6  20.6
26                 16.0  8.8  18.9
eval
first_category_id     4   16    29
week                              
26                 17.6  9.7  23.9
27                 16.0  8.3  19.0


**rolling-origin backtest**: each fold moves the cut-off one week back, so every test week is predicted using only data from before it.

In [15]:
PARAMS = dict(num_leaves=63, min_child_samples=100, learning_rate=0.05,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              random_state=42, verbose=-1)
N_TREES = {cat: m.best_iteration_ for cat, m in lgbm_models.items()}

hist = pd.concat([train, val]).sort_values("dt")
last_day = hist["dt"].max()
hist["week"] = (last_day - hist["dt"]).dt.days // 7     # 0 = last week, 1 = week before, ...

def run_fold(test_week):
    tr  = hist[hist["week"] >= test_week + 2]
    cal = hist[hist["week"] == test_week + 1]
    te  = hist[hist["week"] == test_week]
    rows = []
    for cat, tau in TAU.items():
        model = lgb.LGBMRegressor(objective="quantile", alpha=tau,
                                  n_estimators=N_TREES[cat], **PARAMS)
        model.fit(tr[FEATURES], tr[TARGET])

        c = cal[cal["first_category_id"] == cat]
        t = te[te["first_category_id"] == cat]
        shift = conformal_correction(c[TARGET].to_numpy(), model.predict(c[FEATURES]), tau)

        y = t[TARGET].to_numpy()
        q_raw = model.predict(t[FEATURES]).clip(min=0)
        q_conf = (q_raw + shift).clip(min=0)

        for name, q in [("LightGBM", q_raw), ("LightGBM + conformal", q_conf)]:
            rows.append({"test_week": t["dt"].min().date(),
                         "category": retail_configs.CATEGORY_CONFIG[cat]["category_name"],
                         "tau": tau, "model": name, "shift": shift,
                         "coverage": (y <= q).mean(),
                         "pinball": pinball(y, q, tau)})
    return rows

backtest = pd.DataFrame([row for w in range(5) for row in run_fold(w)])

In [16]:
summary = (backtest.groupby(["category", "model"])
           .agg(coverage_mean=("coverage", "mean"),
                coverage_min=("coverage", "min"),
                coverage_max=("coverage", "max"),
                pinball_mean=("pinball", "mean"))
           .round(3))
summary

coverage_mean  coverage_min  coverage_max  \
category   model                                                             
bread      LightGBM                      0.546         0.479         0.624   
           LightGBM + conformal          0.588         0.509         0.673   
meat       LightGBM                      0.720         0.663         0.745   
           LightGBM + conformal          0.736         0.672         0.765   
vegetables LightGBM                      0.868         0.810         0.904   
           LightGBM + conformal          0.860         0.805         0.902   

                                 pinball_mean  
category   model                               
bread      LightGBM                     3.551  
           LightGBM + conformal         3.516  
meat       LightGBM                     1.239  
           LightGBM + conformal         1.235  
vegetables LightGBM                     1.184  
           LightGBM + conformal         1.189

In [18]:
eval_.to_parquet("data/eval_predictions.parquet", index=False)